# Scene Flow full-corpus downloader (Modal-friendly)

Downloads the **full Scene Flow** dataset (Driving + Monkaa + FlyingThings3D, finalpass RGB + dense disparity, **~205 GB total**) from the official Freiburg LMB server into a persistent location.

**Designed for Modal.com**: when run on a Modal notebook with a Volume mounted at `/data` (or wherever you choose), the downloaded files persist across sessions and can be mounted into training jobs later. The notebook also runs unchanged on a plain Jupyter / Colab / local machine — just set `TARGET` to whatever path you want.

## Safety design

- **One cell per file.** If a download fails or stalls, you can re-run   just that one cell.
- **Resume on re-run.** Every download uses `wget -c` (continue), so   re-running a cell after an interruption picks up from the last byte.
- **Size verification.** After each download, the file size is checked   against published expected ranges and prints a loud warning on   mismatch.
- **No early bail.** A failed download in one cell does not prevent   others from proceeding.
- **No deletion.** Downloaded tarballs are never auto-removed; you   decide when to extract or discard them.

## Files downloaded

| subset | file | approx. size |
|---|---|---|
| Driving | `driving__frames_finalpass.tar` | 6.5 GB |
| Driving | `driving__disparity.tar.bz2` | 9.5 GB |
| Monkaa | `monkaa__frames_finalpass.tar` | 38 GB |
| Monkaa | `monkaa__disparity.tar.bz2` | 13 GB |
| FlyingThings3D | `flyingthings3d__frames_finalpass.tar` | 45 GB |
| FlyingThings3D | `flyingthings3d__disparity.tar.bz2` | 93 GB |
| **total** | | **~205 GB** |

Freiburg's outbound bandwidth varies; expect 1–10 MB/s per stream. Allow several hours for the whole corpus (FlyingThings3D disparity alone can take 6+ hours on a slow connection).


## 1. Setup — mount target directory and verify space

In [ ]:
import os, shutil, subprocess
from pathlib import Path

# ---- TARGET DIRECTORY ----
# On Modal: set this to your mounted Volume path (e.g. '/data/sceneflow').
# On other Jupyter envs: any writable path with ~210 GB free.
TARGET = Path('/data/sceneflow')

TARGET.mkdir(parents=True, exist_ok=True)

# Disk-space sanity check
stat = shutil.disk_usage(TARGET)
free_gb = stat.free / 1e9
needed_gb = 210
print(f'TARGET = {TARGET}')
print(f'free disk space at TARGET: {free_gb:.1f} GB')
print(f'estimated required: {needed_gb} GB (downloads only; +200 GB for extracted)')
if free_gb < needed_gb:
    print(f'\n!! WARNING: only {free_gb:.0f} GB free; downloading the '
          f'full corpus needs ~{needed_gb} GB. Make sure your Modal '
          f'Volume is large enough or change TARGET.')

# Verify wget is installed (it should be on every Linux container)
rc = subprocess.call(['which', 'wget'], stdout=subprocess.DEVNULL)
if rc != 0:
    print('!! wget missing — install it now:')
    !apt-get -qq install -y wget


## 2. Helper functions

In [ ]:
import time, subprocess
from pathlib import Path


def fmt_bytes(b):
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if b < 1024:
            return f'{b:.2f} {unit}'
        b /= 1024
    return f'{b:.2f} PB'


def expected_status(path: Path, lo: int, hi: int) -> str:
    if not path.exists():
        return 'missing'
    sz = path.stat().st_size
    if sz < lo:
        return f'partial ({fmt_bytes(sz)} of expected ~{fmt_bytes(lo)})'
    if sz > hi * 1.1:  # generous upper-bound fudge
        return f'oversize ({fmt_bytes(sz)})'
    return f'OK ({fmt_bytes(sz)})'


def download_one(url: str, target_dir: Path, lo_bytes: int, hi_bytes: int,
                  max_attempts: int = 50, retry_sleep: int = 60) -> bool:
    fname = url.rsplit('/', 1)[-1]
    out = target_dir / fname
    print(f'\n=== {fname} ===')
    print(f'url:    {url}')
    print(f'target: {out}')
    status = expected_status(out, lo_bytes, hi_bytes)
    print(f'status before: {status}')
    if status.startswith('OK'):
        print('already complete — skipping.')
        return True
    attempt = 1
    while attempt <= max_attempts:
        print(f'\n--- attempt {attempt}/{max_attempts}, '
              f'started {time.strftime("%H:%M:%S")} ---')
        cmd = [
            'wget', '-c',
            '--tries=0',
            '--waitretry=30',
            '--timeout=120',
            '--read-timeout=300',
            '--retry-connrefused',
            '--show-progress',
            '--progress=bar:force:noscroll',
            '-O', str(out), url,
        ]
        rc = subprocess.call(cmd)
        status = expected_status(out, lo_bytes, hi_bytes)
        print(f'status after attempt: {status}  (wget rc={rc})')
        if status.startswith('OK'):
            print(f'!! download complete: {fname}')
            return True
        attempt += 1
        if attempt <= max_attempts:
            print(f'sleeping {retry_sleep}s before retry...')
            time.sleep(retry_sleep)
    print(f'!! gave up on {fname} after {max_attempts} attempts.')
    return False

print('helpers ready.')


## 3. Connectivity & permission check

Quick HEAD request against Freiburg before committing to a 200 GB download. If this fails, your environment cannot reach the data and no further cell will succeed.

In [ ]:
test_url = (
    'https://lmb.informatik.uni-freiburg.de/data/'
    'SceneFlowDatasets_CVPR16/Release_april16/data/'
    'Driving/raw_data/driving__frames_finalpass.tar'
)
import subprocess
out = subprocess.run(
    ['wget', '--spider', '--server-response', '--max-redirect=2',
     '--timeout=20', '-q', '-O', '-', test_url],
    capture_output=True, text=True)
print('rc:', out.returncode)
print('--- server response ---')
print(out.stderr[-1500:])
if out.returncode == 0:
    print('\n[OK] connectivity to Freiburg looks good. Proceed.')
else:
    print('\n[FAIL] cannot reach Freiburg. Check Modal egress / DNS / '
          'IP allowlist before starting downloads.')


## 4. Download `driving__frames_finalpass.tar`

**Subset:** Driving  
**Kind:** frames_finalpass  
**Approx. size:** 6--7 GB  

Re-running this cell after a partial download will resume from the last byte (wget `-c`).

In [ ]:
download_one(
    url='https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/raw_data/driving__frames_finalpass.tar',
    target_dir=TARGET,
    lo_bytes=6500000000,
    hi_bytes=7300000000,
    max_attempts=50,
    retry_sleep=60,
)


## 5. Download `driving__disparity.tar.bz2`

**Subset:** Driving  
**Kind:** disparity  
**Approx. size:** 9--10 GB  

Re-running this cell after a partial download will resume from the last byte (wget `-c`).

In [ ]:
download_one(
    url='https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Driving/derived_data/driving__disparity.tar.bz2',
    target_dir=TARGET,
    lo_bytes=9000000000,
    hi_bytes=10500000000,
    max_attempts=50,
    retry_sleep=60,
)


## 6. Download `monkaa__frames_finalpass.tar`

**Subset:** Monkaa  
**Kind:** frames_finalpass  
**Approx. size:** 35--42 GB  

Re-running this cell after a partial download will resume from the last byte (wget `-c`).

In [ ]:
download_one(
    url='https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Monkaa/raw_data/monkaa__frames_finalpass.tar',
    target_dir=TARGET,
    lo_bytes=35000000000,
    hi_bytes=42000000000,
    max_attempts=50,
    retry_sleep=60,
)


## 7. Download `monkaa__disparity.tar.bz2`

**Subset:** Monkaa  
**Kind:** disparity  
**Approx. size:** 11--16 GB  

Re-running this cell after a partial download will resume from the last byte (wget `-c`).

In [ ]:
download_one(
    url='https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/Monkaa/derived_data/monkaa__disparity.tar.bz2',
    target_dir=TARGET,
    lo_bytes=11000000000,
    hi_bytes=16000000000,
    max_attempts=50,
    retry_sleep=60,
)


## 8. Download `flyingthings3d__frames_finalpass.tar`

**Subset:** FlyingThings3D  
**Kind:** frames_finalpass  
**Approx. size:** 45--50 GB  

Re-running this cell after a partial download will resume from the last byte (wget `-c`).

In [ ]:
download_one(
    url='https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/raw_data/flyingthings3d__frames_finalpass.tar',
    target_dir=TARGET,
    lo_bytes=45000000000,
    hi_bytes=50000000000,
    max_attempts=50,
    retry_sleep=60,
)


## 9. Download `flyingthings3d__disparity.tar.bz2`

**Subset:** FlyingThings3D  
**Kind:** disparity  
**Approx. size:** 90--100 GB  

Re-running this cell after a partial download will resume from the last byte (wget `-c`).

In [ ]:
download_one(
    url='https://lmb.informatik.uni-freiburg.de/data/SceneFlowDatasets_CVPR16/Release_april16/data/FlyingThings3D/derived_data/flyingthings3d__disparity.tar.bz2',
    target_dir=TARGET,
    lo_bytes=90000000000,
    hi_bytes=100000000000,
    max_attempts=50,
    retry_sleep=60,
)


## 10. Final summary

Re-checks every file, prints sizes, and reports total disk used.

In [ ]:
FILES = [
    ('driving__frames_finalpass.tar', 6500000000, 7300000000),
    ('driving__disparity.tar.bz2', 9000000000, 10500000000),
    ('monkaa__frames_finalpass.tar', 35000000000, 42000000000),
    ('monkaa__disparity.tar.bz2', 11000000000, 16000000000),
    ('flyingthings3d__frames_finalpass.tar', 45000000000, 50000000000),
    ('flyingthings3d__disparity.tar.bz2', 90000000000, 100000000000),
]

total_bytes = 0
all_ok = True
print(f'{"file":<55s} {"status":<35s} {"size":>15s}')
print('-' * 110)
for fname, lo, hi in FILES:
    p = TARGET / fname
    st = expected_status(p, lo, hi)
    sz = p.stat().st_size if p.exists() else 0
    total_bytes += sz
    if not st.startswith('OK'):
        all_ok = False
    print(f'{fname:<55s} {st:<35s} {fmt_bytes(sz):>15s}')
print('-' * 110)
print(f'TOTAL: {fmt_bytes(total_bytes)}')
print()
if all_ok:
    print('[OK] all 6 tarballs present and within expected size ranges.')
else:
    print('[INCOMPLETE] some files are missing or partial. Re-run the '
          'cells flagged above.')


## 11. (Optional) Extract

Extracting the full corpus consumes about another **200 GB** on top of the tarballs. Don't run this unless you have space — many users keep only the tars in the Modal Volume and extract on-demand inside their training job. Each tar/bz2 has its own cell so partial extraction is fine.

**Warning:** the FlyingThings3D disparity bz2 is the slowest to decompress (it can take 30+ minutes single-threaded).

In [ ]:
import subprocess, shutil
from pathlib import Path

stat = shutil.disk_usage(TARGET)
free_gb = stat.free / 1e9
print(f'free disk before extract: {free_gb:.1f} GB')
if free_gb < 220:
    raise RuntimeError(
        f'only {free_gb:.0f} GB free — extracting the full corpus needs '
        f'~200 GB more. Free up space or skip this cell.')

for tar_name in [
    'driving__frames_finalpass.tar',
    'driving__disparity.tar.bz2',
    'monkaa__frames_finalpass.tar',
    'monkaa__disparity.tar.bz2',
    'flyingthings3d__frames_finalpass.tar',
    'flyingthings3d__disparity.tar.bz2',
]:
    tar_path = TARGET / tar_name
    if not tar_path.exists():
        print(f'[SKIP] {tar_name} not present')
        continue
    print(f'\n=== extracting {tar_name} ===')
    flag = 'xjf' if tar_name.endswith('.bz2') else 'xf'
    rc = subprocess.call(['tar', flag, str(tar_path), '-C', str(TARGET)])
    print(f'rc={rc}  (0 = ok)')

print('\n--- top-level tree under TARGET ---')
for p in sorted(TARGET.iterdir()):
    if p.is_dir():
        print(f'  {p.name}/')


---

### Notes

- **Modal Volume mount**: configure your Modal Notebook with `modal.Volume.from_name('sceneflow', create_if_missing=True)` mounted at `/data/sceneflow`. The volume persists across sessions; downloads survive notebook shutdown.

- **Auto-shutdown**: set the notebook's idle timeout to at least 8 hours if you plan to leave the FlyingThings3D download running unattended.

- **Resuming**: any cell can be re-run safely. If a download was 80% complete when the kernel died, re-running its cell will continue from byte 80%, not restart from zero.

- **After completion**: in your downstream training job (also on Modal or wherever), mount the same Volume read-only and point your data loader at the extracted directories.

- **What's NOT included**: optical flow, disparity_change, cleanpass RGB, motion/depth-boundary weights, occlusion masks. Add cells for those if you need them — same URL pattern, just swap the tar name.
